In [ ]:
def send_email(user_email, test_title, status):
    display_name = Header('수명모델링팀', 'utf-8').encode()
    from_email = formataddr((display_name, 'blmt2024@lgensol.com'))
    user_email = 'hyukjun_jo@lgensol.com'

    send_mail(
        subject = 'Test 업로드 알림',
        message = f'요청하신 Test {test_title}이 업로드되었습니다. \n 10.99.212.69:8443 에서 확인하세요.',
        from_email = from_email,
        recipient_list=[user_email],
        fail_silently=False,
    )

In [8]:
user_email = 'hyukjun_jo@lgensol.com'
title = 'dsds'
send_email(user_email, title, "완료")

In [5]:
from django.utils import timezone
from email.header import Header
from email.utils import formataddr
from django.core.mail import send_mail
from django.http import JsonResponse
# @required_POST
def send_auth_code(request):
    account_id   = request.POST.get('username', '').strip()       # form의 name="username"
    team         = request.POST.get('userteam', '').strip()       # form의 name="userteam"
    member_name  = request.POST.get('memberSelect', '').strip()   # form의 name="memberSelect"
   
    display_name = Header('수명모델링팀', 'utf-8').encode()
    from_email = formataddr((display_name, 'blmt2024@lgensol.com'))
    if not (account_id and team and member_name):
        return HttpResponseBadRequest('아이디, 팀, 이름을 모두 선택해주세요.')
 
    try:
        user = Accounts.objects.using('accounts_db').get(
            id=account_id,
            team=team,
            user=member_name,
            is_active=True
        )
 
    except Accounts.DoesNotExist:
        return JsonResponse({'error':'일치하는 계정이 없습니다.'}, status=400)
 
    code = f"{random.randint(0,999999):06d}"
    request.session['auth_code'] = code
    print(code)
    print(request.session['auth_code'])
    request.session['auth_code_time'] = timezone.now().isoformat()
 
    send_mail(
        subject = '비밀번호 변경 인증번호입니다.',
        message = f'인증번호: {code} (5분간 유효)',
        from_email = from_email,
        recipient_list=[user.email],
        fail_silently=False,
    )
 
    return JsonResponse({'message': f'{user.email}로 인증번호를 보냈습니다.'})

In [7]:
from django.conf import settings

if not settings.configured:
    settings.configure(
        # 필수 설정: 어떤 이메일 백엔드를 사용할 것인지 지정
        # 이메일 전송 테스트 중이라면 'django.core.mail.backends.console.EmailBackend' 사용
        EMAIL_BACKEND='django.core.mail.backends.smtp.EmailBackend', 
        
        # SMTP 서버 설정 (예: GMail, AWS SES 등)
        EMAIL_HOST="spam.lgensol.com",             # 환경 변수에서 가져옵니다.
        EMAIL_PORT=25,  # 포트 번호 (일반적으로 587 또는 465)
        EMAIL_USE_TLS=False,                            # TLS 사용 여부 (일반적으로 True)
    )


In [14]:
from email.header import Header
from email.utils import formataddr
from django.core.mail import send_mail

display_name = Header('수명모델링팀', 'utf-8').encode()
from_email = formataddr((display_name, 'blmt2024@lgensol.com'))
user_email = 'hyukjun_jo@lgensol.com'

send_mail(
    subject = '비밀번호 변경 인증번호입니다.',
    message = f'인증번호: {code} (5분간 유효)',
    from_email = from_email,
    recipient_list=[user_email],
    fail_silently=False,
)

1